In [2]:
import torch
torch.cuda.is_available()

True

## 스칼라, 벡터, 행렬 텐서

파이토치 텐서는 배열과 유사한 구조를 위한 데이터 컨테이너이다.

스칼라는 0차원 텐서, 벡터는 1차원 텐서, 행렬은 2차원 텐서이고, 고차원 텐서의 경우 특별한 용어가 없으므로 일반적으로 3차원 텐서를 3D 텐서라 부른다.

In [3]:
# 파이썬 정수로 0차원 텐서(스칼라)를 만든다.
tensor0d = torch.tensor(1)

# 파이썬 리스트로 1차원 텐서(벡터)를 만든다.
tensor1d = torch.tensor([1, 2, 3])

# 중첩된 파이썬 리스트로 2차원 텐서(행렬)를 만든다.
tensor2d = torch.tensor([[1, 2, 3],
                        [4, 5, 6]])

# 중첩된 파이썬 리스트로 3차워 텐서를 만든다.
tensor3d = torch.tensor([[[1, 2], [3, 4]],
                        [[5, 6], [7, 8]]])

# 텐서 출력
print(tensor2d)

#.shape 속성으로 텐서 크기 확인 (2개의 행과 3개의 열)
print(tensor2d.shape)

#.reshape와 .view 속성으로 이 텐서의 크기를 3 x 2 텐서로 바꾸기
print(tensor2d.reshape(3, 2))
print(tensor2d.view(3, 2)) # 더 많이 사용됨

#.T 속성으로 텐서 전치
print(tensor2d.T)

#.matmul 속성으로 두 행렬 곱하기
print(tensor2d.matmul(tensor2d.T))

# @를 사용하여 행렬 곱하기
print(tensor2d @ tensor2d.T)

tensor([[1, 2, 3],
        [4, 5, 6]])
torch.Size([2, 3])
tensor([[1, 2],
        [3, 4],
        [5, 6]])
tensor([[1, 2],
        [3, 4],
        [5, 6]])
tensor([[1, 4],
        [2, 5],
        [3, 6]])
tensor([[14, 32],
        [32, 77]])
tensor([[14, 32],
        [32, 77]])


## 모델을 계산 그래프로 보기

다음 코드는 간단한 로지스틱 회귀 분류기의 정방향 계산을 구현한 것이다.

이를 하나의 층을 가진 신경망으로 볼 수 있다.

이 모델은 0과 1 사이의 점수를 반환하며, 정답 클래스 레이블(0또는 1)과 비교하여 손실을 계산한다.


In [4]:
import torch.nn.functional as F

y = torch.tensor([1.0]) # 정답 레이블
x1 = torch.tensor([1.1]) # 입력
w1 = torch.tensor([2.2]) # 가중치 파라미터
b = torch.tensor([0.0]) # 바이어스
z = x1 * w1 + b # 순입력
a = torch.sigmoid(z) # 활성화 함수와 출력
loss = F.binary_cross_entropy(a, y)

파이토치의 autograd 엔진은 텐서에 대해 수행되는 모든 연산을 추적하여 백그라운드에서 계산 그래프를 구축한다.

grad 함수를 호출하며 모델 파라미터 $w_1$ 에 대한 손실 그레이디언트를 계산할 수 있다.

In [5]:
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor(1.1)
w1 = torch.tensor([2.2], requires_grad = True)
b = torch.tensor([0.0], requires_grad = True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

# 기본적으로 파이토치는 그레이디언트 계산 후에 메모리를 해제하기 위해 계산 그래프를 삭제한다.
# 하지만 계산 그래프를 금방 재사용하므로 메모리에 유지되도록 retain_grapth=True로 설정한다.
grad_L_w1 = grad(loss, w1, retain_graph = True)
grad_L_b = grad(loss, b, retain_graph=True)

print("grad_L_w1 = ", grad_L_w1)
print("grad_L_b = ", grad_L_b)

grad_L_w1 =  (tensor([-0.0898]),)
grad_L_b =  (tensor([-0.0817]),)


## 다층 신경망 만들기
이제 심층 신경망을 구축하기 위한 라이브러리로서의 파이토치에 초점을 맞추어, 다층 퍼셉프론 또는 완전 연결 신경망을 만들어보자.

파이토치로 신경망을 만들 때 torch.nn.Module 클래스를 상속하여 사용자 정의 신경망 구조를 정의할 수 있다.

이 클래스를 상속한 자식 클래스의 \_\_init__ 생성자에서 신경망을 정의하고 forward 메서드에서 층이 어떻게 상호작용하는지 지정한다. forward 메서드에는 입력 데이터가 신경망을 어떻게 통과하여 계산 그래프로 구성되는지를 기술한다. 이와 달리 backward 메서드를 사용해 모델 파라미터에 대한 손실 함수의 그레이디언트를 계산한다.

다음 코드는 Module 클래스의 사용법을 보여 주기 위해 2개의 은닉층을 가진 전형적인 다층 퍼셉트론을 만든다.

In [6]:
class NeuralNetwork(torch.nn.Module):
  def __init__(self, num_inputs, num_outputs):
    super().__init__()

    self.layers = torch.nn.Sequential(

        # 첫 번째 은닉층
        torch.nn.Linear(num_inputs, 30), # Linear 층은 입력 개수와 출력 크기를 매개변수로 가짐
        torch.nn.ReLU(), # 은닉층 사이에 비선형 함수를 놓음

        # 두 번째 은닉층
        torch.nn.Linear(30, 20), # 은닉층의 출력 노드 개수는 다음 층의 입력 노드 개수와 맞아야 함
        torch.nn.ReLU(),

        # 출력층
        torch.nn.Linear(20, num_outputs),
    )

  def forward(self, x):
    logits = self.layers(x)
    return logits # 마지막 층의 출력을 로짓이라 부름

model = NeuralNetwork(50, 3)
print(model)


NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


## 효율적인 데이터 로더 설정하기
모델을 훈련하기 전에 파이토치 훈련 과정에서 효율적으로 데이터를 순회하는 데 사용하는 데이터 로더에 대해 간략히 논의한다.

Dataset 클래스를 사용해 데이터 레코드가 로드되는 방법을 정의한 객체를 만든다. DataLoader는 데이터 셔플링과 배치로 묶는 방식을 처리한다.

사용자 정의 Dataset 클래스를 구현한다. 이를 사용해 데이터 로더를 만들 때 사용할 훈련 데이터셋과 테스트 데이터셋을 만든다.

5개의 샘플과 2개의 특성을 가진 간단한 데이터셋을 만들어보자. 훈련 샘플과 함께 클래스 레이블을 담은 텐서도 만든다. 3개의 샘플은 클래스 0, 2개의 샘플은 클래스 1에 속한다. 또한 2개의 샘플로 구성된 테스트 세트를 만든다.

In [7]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6]
])
y_test = torch.tensor([0, 1])

그런 다음 파이토치 Dataset 클래스를 상속하여 사용자 정의 데이터셋 클래스 ToyDataset을 만든다.

In [8]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
  def __init__(self, X, y):
    self.features = X
    self.labels = y

  def __getitem__(self, index): # 정확히 하나의 데이터 레코드와 이에 해당하는 레이블을 추출한다
    one_x = self.features[index]
    one_y = self.labels[index]
    return one_x, one_y

  def __len__(self):
    return self.labels.shape[0] # 데이터셋의 총 길이를 반환한다.

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)


ToyDataset 클래스의 목적은 파이토치 DataLoader의 객체를 만드는 것이다.

예제 데이터셋에 사용할 ToyDataset 클래스를 만들었으니 파이토치 DataLoader 클래스를 사용해 데이터를 샘플링해보자.

In [11]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset = train_ds, # 앞서 만든 ToyDataset 객체를 데이터 로더의 입력으로 사용한다.
    batch_size=2,
    shuffle=True, # 데이터를 셔플링할지 여부
    num_workers=0, # 백그라운드 프로세스 개수
    drop_last=True # 실전에서는 훈련 에포크의 마지막 배치 크기가 매우 작다면 훈련 중 수렴을 방해할 수 있으므로 에포크의 마지막 배치를 버림
)

test_loader = DataLoader(
    dataset = test_ds,
    batch_size = 2,
    shuffle = False, # 테스트 데이터셋은 셔플링이 필요하지 않음
    num_workers = 0
)

for idx, (x, y) in enumerate(train_loader):
  print(f"배치 {idx+1}:", x, y)

배치 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
배치 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])


## 일반적인 훈련 루프
예제 데이터셋으로 신경망을 훈련해보자.

훈련 코드는 다음과 같다.

In [14]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs = 2, num_outputs=2)
optimizer = torch.optim.SGD( # 옵티마이저에 최적화할 파라미터를 전달해야 한다.
    model.parameters(), lr = 0.5
)

num_epochs = 3
for epoch in range(num_epochs):

  model.train()
  for batch_idx, (features, labels) in enumerate(train_loader):
    logits = model(features)

    loss = F.cross_entropy(logits, labels)

    optimizer.zero_grad() # 이전 반복에서 구한 그레이디언트를 0으로 지정하여 의도치 않게 그레이디언트가 누적되지 않게 함
    loss.backward() # 모델 파라미터에 대한 손실 그레이디언트를 계산함
    optimizer.step() # 옵티마이저가 그레이디언트를 사용해 모델 파라미터를 업데이트 함

    ## 로깅
    print(f"에포크: {epoch+1:03d}/{num_epochs:03d}"
          f" | 배치 {batch_idx:03d}/{len(train_loader):03d}"
          f" | 훈련 손실: {loss: .2f}")


에포크: 001/003 | 배치 000/002 | 훈련 손실:  0.75
에포크: 001/003 | 배치 001/002 | 훈련 손실:  0.65
에포크: 002/003 | 배치 000/002 | 훈련 손실:  0.44
에포크: 002/003 | 배치 001/002 | 훈련 손실:  0.13
에포크: 003/003 | 배치 000/002 | 훈련 손실:  0.03
에포크: 003/003 | 배치 001/002 | 훈련 손실:  0.00


In [15]:
# 예측 정확도를 계산하기 위한 함수
def compute_accuracy(model, dataloader):

  model = model.eval()
  correct = 0.0
  total_examples = 0

  for idx, (features, labels) in enumerate(dataloader):

    with torch.no_grad():
      logits = model(features)

    predictions = torch.argmax(logits, dim=1)
    compare = labels == predictions # 레이블의 일치 여부에 따라 True/False 값의 텐서를 반환함
    correct += torch.sum(compare) # sum 연산은 True 값의 개수를 카운트 함
    total_examples += len(compare)

  return (correct / total_examples).item() #0~1 사이의 값인 정확한 예측의 비율. .item()은 텐서 값을 파이썬 실숫값으로 반환함

print(compute_accuracy(model, train_loader))
print(compute_accuracy(model, test_loader))

1.0
1.0


## 모델 저장과 로드
모델을 훈련했으니 나중에 재사용하기 위해 모델을 저장하는 방법을 알아보자.



```
torch.save(model.state_dict(), "model.pth")
```

모델의 state_dict는 모델의 각 층과 훈련 가능한 파라미터(가중치와 편향)를 매핑한 파이썬 딕셔너리 객체이다. "model.pth"는 디스크에 저장하기 위해 필요한 파일 이름이며 자유롭게 지정할 수 있다. 원하는 이름과 확장자를 지정할 수 있지만 .pth나 .pt가 가장 널리 사용되는 확장자이다.

모델을 저장하고 난 후에는 디스크에서 불러올 수 있다.


```
model = NueralNetwork(2, 2)
model.load_state_dict(torch.load("model.pth"))
```

torch.load("model.pth")는 파일 "model.path"를 읽고 모델 파라미터를 담고 있는 파이썬 딕셔너리 객체를 재구성한다. model.load_state_dict()는 이 파라미터를 모델에 적용하여 저장했던 학습된 상태를 복원한다.



#

## GPU로 훈련 성능 최적화하기
일반적인 CPU에 비해 심층 신경망의 훈련 속도를 높이기 위해 GPU를 활용하는 방법을 알아보자.

In [16]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs = 2, num_outputs=2)

device = torch.device("cuda") # GPU로 장치 변수를 정의한다.
model = model.to(device) # 모델을 GPU로 전송한다.

optimizer = torch.optim.SGD( # 옵티마이저에 최적화할 파라미터를 전달해야 한다.
    model.parameters(), lr = 0.5
)

num_epochs = 3
for epoch in range(num_epochs):

  model.train()
  for batch_idx, (features, labels) in enumerate(train_loader):
    features, labels = features.to(device), labels.to(device) # 데이터를 GPU로 전송한다.
    logits = model(features)

    loss = F.cross_entropy(logits, labels)

    optimizer.zero_grad() # 이전 반복에서 구한 그레이디언트를 0으로 지정하여 의도치 않게 그레이디언트가 누적되지 않게 함
    loss.backward() # 모델 파라미터에 대한 손실 그레이디언트를 계산함
    optimizer.step() # 옵티마이저가 그레이디언트를 사용해 모델 파라미터를 업데이트 함

    ## 로깅
    print(f"에포크: {epoch+1:03d}/{num_epochs:03d}"
          f" | 배치 {batch_idx:03d}/{len(train_loader):03d}"
          f" | 훈련 손실: {loss: .2f}")

에포크: 001/003 | 배치 000/002 | 훈련 손실:  0.75
에포크: 001/003 | 배치 001/002 | 훈련 손실:  0.65
에포크: 002/003 | 배치 000/002 | 훈련 손실:  0.44
에포크: 002/003 | 배치 001/002 | 훈련 손실:  0.13
에포크: 003/003 | 배치 000/002 | 훈련 손실:  0.03
에포크: 003/003 | 배치 001/002 | 훈련 손실:  0.00
